# WikiDT Phase 7 — Modular OCR Embeddings + Pluggable QA Reader

**Goal:** decouple OCR ingestion and embeddings from the final question-answering model. The same cached OCR passage embeddings can support retrieval, classification, clustering, or future QA readers. This experiment also tests **hybrid semantic + lexical retrieval** and **structure-aware OCR text**.

**Important:** Amazon Textract produces text and coordinates, **not BERT embeddings**. We create independent 384-dimensional semantic passage embeddings using `sentence-transformers/all-MiniLM-L6-v2`. A downstream BERT extractive QA reader still consumes **question + text passage** (not these MiniLM vectors directly). Different encoder spaces are not interchangeable. The reusable embedding interface is independent of the QA reader.

**Research integrity:** Phase 7 results are unknown until run. Compare retrieval recall and held-out EM/F1 with Phase 6 on the **same manifest** when possible. Validation determines checkpoints; test is only evaluated after training.

## 1 — Dependencies and configuration

In [1]:
import os
os.environ['USE_TF']='0';os.environ['USE_FLAX']='0';os.environ['TRANSFORMERS_NO_TF']='1'
%pip -q install 'transformers>=4.41,<5' 'huggingface_hub>=0.24' 'sentence-transformers>=3,<6' scikit-learn pandas pyarrow tqdm accelerate


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2 — Setup and answer normalization

Keep the same normalization, reproducible seed, and dataset sizes as Phase 6. Drive is used for persistent OCR and embedding caches.

In [2]:
import os, re, json, ast, time, random, tarfile, shutil, unicodedata, hashlib, collections, traceback
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from IPython.display import display
from huggingface_hub import HfApi, hf_hub_download
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
print('[02] Mounting Drive (if running in Colab)')
try:
 from google.colab import drive
 drive.mount('/content/drive')
except ImportError: print('[02] Not in Colab; using local paths')
SIZES={'train':300,'val':60,'test':60} # start with 24/8/8 if preferred; then 450/150/150
SEED=42; EPOCHS=4; BATCH_SIZE=4; LR=2e-5
GRAD_ACCUM=2; WARMUP_RATIO=0.1; PATIENCE=2; NEGATIVE_PER_QUESTION=2
TOP_K=6; CHUNK_WORDS=140; CHUNK_STEP=95; MAX_LEN=384; DOC_STRIDE=96
EMBED_MODEL='sentence-transformers/all-MiniLM-L6-v2'; SEMANTIC_WEIGHT=0.55; LEXICAL_WEIGHT=0.45
USE_STRUCTURED_OCR=True; REUSE_PHASE6_MANIFEST=True
MODEL_ID='deepset/bert-base-cased-squad2'; REPO='AmazonScience/WikiDT'; REV='main'
FORCE_RETRAIN=False; USE_PHASE4_COHORT_IF_AVAILABLE=False
ROOT=Path('/content/drive/MyDrive/WikiDT_Persistent') if Path('/content/drive/MyDrive').exists() else Path('/mnt/data/WikiDT_Persistent')
RUN=ROOT/'phase7_modular_ocr_embeddings'/'semantic_hybrid_v1'/('pilot_'+ '_'.join(str(SIZES[s]) for s in ('train','val','test')))
RUN.mkdir(parents=True,exist_ok=True)
LOCAL=Path('/content/wikidt_phase7_cache') if Path('/content').exists() else Path('/mnt/data/wikidt_phase5_cache')
LOCAL.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
def log(section,msg): print(f'[{section}][{time.strftime("%H:%M:%S")}] {msg}',flush=True)
# One canonical answer conversion shared by sampling, span labeling, audit, and scoring.
MISSING_ANSWERS={'', 'none','no answer','noanswer','nan','null','n/a','[none]','[null]'}
def answer_text(value):
    if value is None: return ''
    if isinstance(value,(list,tuple)):
        return answer_text(value[0]) if len(value)==1 else ''
    if isinstance(value,(float,np.floating)) and np.isnan(value): return ''
    s=str(value).strip()
    if s.startswith('[') and s.endswith(']'):
        try:
            v=ast.literal_eval(s)
            if isinstance(v,(list,tuple)):
                return answer_text(v[0]) if len(v)==1 else ''
        except (ValueError,SyntaxError):
            pass
        if re.fullmatch(r'\[\s*(none|null|nan)\s*\]',s,flags=re.I):return ''
    return '' if s.casefold() in MISSING_ANSWERS else s
def norm(x):
    return ' '.join(unicodedata.normalize('NFKC',answer_text(x)).casefold().split())
def eligible(x):
    a=answer_text(x)
    return bool(a and norm(a) not in MISSING_ANSWERS and any(ch.isalpha() for ch in a))
assert answer_text("['hawthorn']")=='hawthorn'
assert not eligible('[None]') and not eligible('None') and not eligible(['None'])
assert eligible("['tipperary(1950-1959)']")
log('02','Answer normalization self-tests passed; list wrappers and [None] handled')
log('02',f'Device={DEVICE}, sizes={SIZES}, ROOT={ROOT}, RUN={RUN}, local HF cache={LOCAL}')
log('02',f'Free local disk: {shutil.disk_usage(LOCAL).free/1e9:.2f} GB')

[02] Mounting Drive (if running in Colab)
Mounted at /content/drive
[02][03:19:46] Answer normalization self-tests passed; list wrappers and [None] handled
[02][03:19:46] Device=cpu, sizes={'train': 300, 'val': 60, 'test': 60}, ROOT=/content/drive/MyDrive/WikiDT_Persistent, RUN=/content/drive/MyDrive/WikiDT_Persistent/phase7_modular_ocr_embeddings/semantic_hybrid_v1/pilot_300_60_60, local HF cache=/content/wikidt_phase7_cache
[02][03:19:46] Free local disk: 242.19 GB


## 3 — Inspect the official Hugging Face WikiDT repository

**Purpose:** Discover the **actual** paths to WikiDT sample metadata and Amazon Textract OCR, rather than guessing local file names.

**Input:** Dataset repository `AmazonScience/WikiDT` and revision `main`. **Output:** `SAMPLE_FILES` (train/val/test metadata paths) and `OCR_ARCHIVE` (the `ocr.tar` path).

**What happens:** The Hugging Face API lists repository files; the notebook checks that each required file is found exactly once. This is metadata discovery, **not** the large OCR download.

**Expected output:** Repository file count, example paths, selected metadata paths, and the archive path.

**Debugging:** An error about missing or multiple matches means the upstream repository layout differs from the notebook's assumptions. Inspect the printed repository file list before changing path logic. Network/authentication/rate-limit errors are distinct from path-matching errors.

In [3]:
t=time.time(); log('03','Listing official Hugging Face dataset files')
api=HfApi(); files=api.list_repo_files(REPO,repo_type='dataset',revision=REV)
log('03',f'Repository has {len(files)} files; first 25: {files[:25]}')
def find_file(suffix):
 matches=[f for f in files if f==suffix or f.endswith('/'+suffix)]
 if len(matches)!=1:raise RuntimeError(f'Expected exactly one {suffix}; matches={matches[:20]}')
 return matches[0]
SAMPLE_FILES={s:find_file('samples/'+s+'.json') for s in SIZES}
OCR_ARCHIVE=find_file('ocr.tar')
log('03',f'Sample files={SAMPLE_FILES}; OCR archive={OCR_ARCHIVE}; elapsed={time.time()-t:.1f}s')

[03][03:19:46] Listing official Hugging Face dataset files
[03][03:19:46] Repository has 28 files; first 25: ['.gitattributes', 'README.md', 'WikiDT-dataset/WikiTableExtraction/detection.partaa', 'WikiDT-dataset/WikiTableExtraction/detection.partab', 'WikiDT-dataset/WikiTableExtraction/detection.partac', 'WikiDT-dataset/WikiTableExtraction/detection.partad', 'WikiDT-dataset/WikiTableExtraction/detection.partae', 'WikiDT-dataset/WikiTableExtraction/detection.partaf', 'WikiDT-dataset/WikiTableExtraction/detection.partag', 'WikiDT-dataset/WikiTableExtraction/structure.partaa', 'WikiDT-dataset/WikiTableExtraction/structure.partab', 'WikiDT-dataset/WikiTableExtraction/structure.partac', 'WikiDT-dataset/WikiTableExtraction/structure.partad', 'WikiDT-dataset/WikiTableExtraction/structure.partae', 'WikiDT-dataset/images.partaa', 'WikiDT-dataset/images.partab', 'WikiDT-dataset/images.partac', 'WikiDT-dataset/images.partad', 'WikiDT-dataset/images.partae', 'WikiDT-dataset/images.partaf', 'WikiDT

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 4 — Build a small, reproducible QA cohort

**Purpose:** Select the questions to train and evaluate on, using official WikiDT metadata and only questions with associated Textract file references.

**Input:** Hugging Face `samples/{split}.json` metadata, plus optional previously saved Phase 4 cohort Parquet files in Drive.

**What happens:**
1. Parse sample JSON or JSONL records.
2. Normalize the ground-truth answer, reject missing/non-alphabetic targets, and require a nonempty question and `all_ocr_files_textract` references.
3. Try reusing a Phase 4 cohort **only if every selected record uniquely matches official WikiDT metadata and OCR references**; otherwise sample from Hugging Face with a fixed seed.
4. Deduplicate by image basename, normalized question, and normalized answer across splits.
5. Construct `qa`, with `split`, `image`, `question`, `answer`, `ocr_files`, and `cohort_source`.

**Expected output:** Raw and selected counts for each split, data source, and unique OCR reference counts. Review a few rows to ensure the questions, answers, and OCR paths make sense.

**Debugging:** If fewer eligible rows than requested are available, reduce `SIZES` or inspect metadata fields. **Caution:** This is not a guaranteed page-disjoint split; the later diagnostics report page overlap.

In [4]:
t=time.time(); log('04','Loading HF sample metadata (not OCR archive)')
def records(path):
 raw=Path(path).read_text(encoding='utf-8')
 try: obj=json.loads(raw)
 except json.JSONDecodeError:obj=[json.loads(line) for line in raw.splitlines() if line.strip()]
 if isinstance(obj,list):return obj
 if isinstance(obj,dict):
  for k in ('samples','data','records','items'):
   if isinstance(obj.get(k),list):return obj[k]
  if all(isinstance(v,dict) for v in obj.values()):return list(obj.values())
 raise ValueError('Unsupported sample JSON structure: '+str(type(obj)))
def safe_ref(x):
 p=str(x).replace('\\','/').lstrip('./')
 if 'ocr/textract/' in p:p='ocr/textract/'+p.split('ocr/textract/',1)[1]
 if '..' in Path(p).parts:raise ValueError('Unsafe path '+p)
 return p
def refs(r):
 v=r.get('all_ocr_files_textract') or []
 if isinstance(v,str):v=[v]
 return sorted(set(safe_ref(x) for x in v if isinstance(x,str) and 'textract/' in x.replace('\\','/')))
def key(r):return (Path(str(r.get('image',''))).name.casefold(),norm(r.get('question','')),norm(answer_text(r.get('answer',''))))
manifest=[]
used_keys=set()
# Phase 7 independently samples the same deterministic cohort as Phase 6 by default.
# To ensure identical evaluation rows, verify manifest hashes against Phase 6.

for split,count in SIZES.items():
 path=hf_hub_download(repo_id=REPO,repo_type='dataset',filename=SAMPLE_FILES[split],revision=REV,cache_dir=str(LOCAL))
 raw=[r for r in records(path) if isinstance(r,dict)]
 log('04',f'{split}: HF records={len(raw)}; sample keys={list(raw[0])[:20] if raw else []}')
 idx=collections.defaultdict(list)
 for r in raw:idx[key(r)].append(r)
 saved_candidates=[ROOT/'embeddings'/'vit_bert_qa_baseline'/'phase4_alpha_mixed_v1_450_150_150'/f'{split}_fixed_qa.parquet',ROOT/'embeddings'/'vit_bert_qa_baseline'/'phase4_alpha_only_v1_450_150_150'/f'{split}_fixed_qa.parquet']
 chosen=[];source='HF fresh'
 if USE_PHASE4_COHORT_IF_AVAILABLE:
  for cp in saved_candidates:
   if cp.exists():
    old=pd.read_parquet(cp)
    if len(old)>=count:
     trial=old.sample(n=count,random_state=SEED) if len(old)>count else old
     trial_keys=[(Path(str(r.image)).name.casefold(),norm(r.question),norm(answer_text(r.answer))) for r in trial.itertuples(index=False)]
     if all(len(idx[k])==1 and refs(idx[k][0]) and eligible(idx[k][0].get('answer')) for k in trial_keys):
      chosen=[idx[k][0] for k in trial_keys] if not any(k in used_keys for k in trial_keys) else [];source=f'Drive Phase4 {cp.name}'
      break
     log('04',f'{split}: existing Phase4 rows not all uniquely matched to HF OCR refs; fallback to HF')
 if not chosen:
  pool=[];seen=set()
  for r in raw:
   k=key(r); a=answer_text(r.get('answer',''))
   if k in seen or k in used_keys or not eligible(a) or not str(r.get('question','')).strip() or not refs(r):continue
   seen.add(k);pool.append(r)
  if len(pool)<count:raise RuntimeError(f'{split}: only {len(pool)} eligible HF records with OCR refs; requested {count}')
  chosen=random.Random(SEED+{'train':0,'val':1,'test':2}[split]).sample(pool,count)
 for r in chosen:
  if key(r) in used_keys: raise RuntimeError('Overlapping sample key across splits')
  used_keys.add(key(r))
  manifest.append(dict(split=split,image=str(r.get('image','')),question=str(r.get('question','')),answer=answer_text(r.get('answer','')),ocr_files=refs(r),cohort_source=source))
 log('04',f'{split}: selected {len(chosen)} questions, source={source}, unique OCR refs={len(set(p for r in chosen for p in refs(r)))}')
qa=pd.DataFrame(manifest)
assert qa.answer.map(eligible).all(), 'Missing or numeric-only answer slipped into cohort'
log('04',f'Canonical answer checks passed: {len(qa)} eligible questions')
qa.assign(ocr_files=qa.ocr_files.map(json.dumps)).to_parquet(RUN/'qa_manifest.parquet',index=False)
log('04',f'Saved manifest; elapsed={time.time()-t:.1f}s')
display(qa[['split','question','answer','cohort_source']].head(5))

[04][03:19:46] Loading HF sample metadata (not OCR archive)


WikiDT-dataset/samples/train.json:   0%|          | 0.00/77.1M [00:00<?, ?B/s]

[04][03:19:53] train: HF records=39488; sample keys=['subset', 'sample_id', 'table_id', 'question', 'image', 'answer', 'all_table_files_web', 'all_ocr_files_web', 'table_retrieval_file_web', 'ocr_retrieval_file_web', 'all_table_files_textract', 'all_ocr_files_textract', 'table_retrieval_file_textract', 'ocr_retrieval_file_textract', 'sub_page', 'sub_page_retrieved', 'sql_str']
[04][03:19:57] train: selected 300 questions, source=HF fresh, unique OCR refs=1302


WikiDT-dataset/samples/val.json:   0%|          | 0.00/26.2M [00:00<?, ?B/s]

[04][03:20:01] val: HF records=16914; sample keys=['subset', 'sample_id', 'table_id', 'question', 'image', 'answer', 'all_table_files_web', 'all_ocr_files_web', 'table_retrieval_file_web', 'ocr_retrieval_file_web', 'all_table_files_textract', 'all_ocr_files_textract', 'table_retrieval_file_textract', 'ocr_retrieval_file_textract', 'sub_page', 'sub_page_retrieved', 'sql_str']
[04][03:20:03] val: selected 60 questions, source=HF fresh, unique OCR refs=171


WikiDT-dataset/samples/test.json:   0%|          | 0.00/25.5M [00:00<?, ?B/s]

[04][03:20:05] test: HF records=14250; sample keys=['subset', 'sample_id', 'table_id', 'question', 'image', 'answer', 'all_table_files_web', 'all_ocr_files_web', 'table_retrieval_file_web', 'ocr_retrieval_file_web', 'all_table_files_textract', 'all_ocr_files_textract', 'table_retrieval_file_textract', 'ocr_retrieval_file_textract', 'sub_page', 'sub_page_retrieved', 'sql_str']
[04][03:20:06] test: selected 60 questions, source=HF fresh, unique OCR refs=256
[04][03:20:06] Canonical answer checks passed: 420 eligible questions
[04][03:20:06] Saved manifest; elapsed=20.0s


,split,question,answer,cohort_source
0,train,Old Mo's Second Spring was the film title used...,1984(55th),HF fresh
1,train,"what's the camp with estimated deaths of 600,000",bełzec,HF fresh
2,train,What date sent has orion as the constellation?,"july 6, 2003",HF fresh
3,train,Which Third has a Second of agnieszka ogrodnic...,katarzyna wicik,HF fresh
4,train,What is the home team when there were more tha...,fitzroy,HF fresh


## 5 — Download the official Textract OCR archive

**Purpose:** Obtain the **existing Amazon Textract output** from Hugging Face. No OCR model is trained or invoked here.

**Input:** All distinct `ocr_files` paths from `qa` and the discovered Hugging Face `ocr.tar` file.

**What happens:** Query the archive's published size, check local free disk, then download/cache the archive with `hf_hub_download`. This is often the longest initial step. A small pilot reduces the number of JSONs extracted, **not necessarily the archive download size**.

**Expected output:** Number of needed OCR JSON paths, archive size if available, free disk space, cached archive path, elapsed time.

**Debugging:** If disk is insufficient, increase Colab storage or change how the archive is streamed; changing `SIZES` alone will not shrink `ocr.tar`. A Hugging Face download error should be investigated separately from training issues. Drive is intentionally not searched as an OCR source.

In [5]:
t=time.time(); wanted=sorted({p for arr in qa.ocr_files for p in arr})
log('05',f'Need {len(wanted)} unique OCR JSON paths for this pilot')
info=api.repo_info(repo_id=REPO,repo_type='dataset',revision=REV,files_metadata=True)
entry=next((x for x in info.siblings if x.rfilename==OCR_ARCHIVE),None)
size=getattr(entry,'size',None)
free=shutil.disk_usage(LOCAL).free
log('05',f'ocr.tar published size={size/1e9:.2f} GB' if size else 'ocr.tar size unavailable')
log('05',f'Free local disk={free/1e9:.2f} GB')
if size and free < size*1.10:
 raise RuntimeError('Insufficient local disk for Hugging Face ocr.tar. Increase Colab storage; this notebook intentionally does not search Drive for OCR.')
archive=Path(hf_hub_download(repo_id=REPO,repo_type='dataset',filename=OCR_ARCHIVE,revision=REV,cache_dir=str(LOCAL)))
log('05',f'Archive downloaded/cached: {archive}; elapsed={time.time()-t:.1f}s')

[05][03:20:06] Need 1709 unique OCR JSON paths for this pilot
[05][03:20:06] ocr.tar published size=2.85 GB
[05][03:20:06] Free local disk=242.06 GB


WikiDT-dataset/ocr.tar:   0%|          | 0.00/2.85G [00:00<?, ?B/s]

[05][03:20:36] Archive downloaded/cached: /content/wikidt_phase7_cache/datasets--AmazonScience--WikiDT/snapshots/db803a4c2031b7170063fed7bc0c445cb8aec9f6/WikiDT-dataset/ocr.tar; elapsed=29.9s


## 6 — Match and extract the selected OCR JSON files

**Purpose:** Map the sample metadata's OCR references to exact TAR members, without extracting the full archive.

**Input:** Downloaded `ocr.tar` and the set of referenced OCR paths. **Output:** `local_paths`, mapping each official OCR reference to a local JSON file.

**What happens:** Canonicalize paths, scan TAR members, flag ambiguous matches, and copy only matched JSON streams into a hashed local filename. Hashing avoids unsafe paths and name collisions.

**Expected output:** Counts of exact matches, missing paths, ambiguous paths, and extraction progress.

**Debugging:** If many paths are missing, compare sample `all_ocr_files_textract` references with TAR member names. Do not silently substitute a similarly named JSON file; that could attach the wrong table to a question. Missing files can lower the OCR coverage ceiling.

In [6]:
t=time.time(); log('06','Scanning TAR members and selecting exact OCR paths')
ocr_dir=LOCAL/'selected_textract';ocr_dir.mkdir(exist_ok=True)
def canon(x):
 x=str(x).replace('\\','/').lstrip('./')
 return x.split('ocr/textract/',1)[-1] if 'ocr/textract/' in x else x
needed={canon(x):x for x in wanted}
member_map={}; ambiguous=set()
with tarfile.open(archive,'r:*') as tar:
 for member in tqdm(tar,desc='Scan TAR'):
  if not member.isfile():continue
  c=canon(member.name)
  if c in needed:
   if c in member_map:ambiguous.add(c)
   else:member_map[c]=member
 for c in ambiguous:member_map.pop(c,None)
 missing=sorted(set(needed)-set(member_map))
 log('06',f'Exact matched={len(member_map)}, missing={len(missing)}, ambiguous={len(ambiguous)}')
 if missing:log('06',f'First missing refs: {missing[:10]}')
 if ambiguous:log('06',f'First ambiguous refs: {list(ambiguous)[:10]}')
 if not member_map:raise RuntimeError('No OCR paths matched archive. Inspect sample OCR refs and TAR member names; do not guess.')
 for c,m in tqdm(member_map.items(),desc='Extract selected OCR'):
  dest=ocr_dir/hashlib.sha256(c.encode()).hexdigest()[:24]
  if dest.exists() and dest.stat().st_size:continue
  stream=tar.extractfile(m)
  if stream is None:continue
  with open(dest,'wb') as f:shutil.copyfileobj(stream,f)
local_paths={needed[c]:ocr_dir/hashlib.sha256(c.encode()).hexdigest()[:24] for c in member_map}
log('06',f'Selected OCR files on local disk={len(local_paths)}; elapsed={time.time()-t:.1f}s')
example=next(iter(local_paths.values()))
print('[06] Sample OCR JSON prefix:',example.read_text(encoding='utf-8',errors='replace')[:700])

[06][03:20:36] Scanning TAR members and selecting exact OCR paths


Scan TAR: 0it [00:00, ?it/s]

[06][03:20:45] Exact matched=1709, missing=0, ambiguous=0


Extract selected OCR:   0%|          | 0/1709 [00:00<?, ?it/s]

[06][03:20:45] Selected OCR files on local disk=1709; elapsed=9.3s
[06] Sample OCR JSON prefix: [
  [
    0,
    [
      [
        "Full name",
        [
          1035,
          551,
          81,
          22
        ]
      ],
      [
        "Milwaukee Rampage",
        [
          1117,
          551,
          186,
          22
        ]
      ],
      [
        "Nickname(s)",
        [
          1035,
          574,
          81,
          23
        ]
      ],
      [
        "Rampage",
        [
          1117,
          574,
          186,
          23
        ]
      ],
      [
        "Founded",
        [
          1035,
          597,
          81,
          22
        ]
      ],
      [
        "1993",
        [
          1117,
          597,
          186,
          22



## 7 — Parse Textract into readable text and audit OCR coverage

**Purpose:** Convert Textract JSON annotations into a text corpus for each question and reuse any previously parsed OCR text.

**Input:** Extracted OCR JSONs and existing Drive `ocr_text_cache.parquet`, if present. **Output:** Text mapped to OCR references, plus question-level OCR availability and answer-coverage diagnostics.

**What happens:** Prefer Textract `LINE` blocks (rather than indiscriminately concatenating `WORD` and `LINE` blocks, which would duplicate text). Assemble the OCR text associated with each selected question. Check whether the normalized ground-truth answer appears anywhere in its OCR text.

**Expected output:** Number of OCR documents parsed or reused, examples of recognized text, and coverage statistics.

**Interpretation:** **OCR answer coverage** measures whether the answer string is present in available recognized text. It is an upper bound for a purely extractive model using that text; it is not the accuracy of BERT. Exact string coverage may undercount recoverable cases when punctuation or spacing differs.

**Debugging:** If coverage is unexpectedly low, inspect raw JSON structure, OCR paths, and answer normalization before training. Flattened `LINE` text may lose table geometry.

## 3 — OCR parsing with spatial layout preservation

The original Phase 6 parser flattened recognized strings. This version uses bounding boxes (when present) to group text into approximate horizontal rows, preserving nearby values. It does **not** claim to reconstruct actual table-cell relationships; it is a heuristic. For plain Textract LINE blocks, it preserves line text. If coordinates are unavailable, it falls back to source order.

Save both readable OCR text and an audit of which documents have coordinates.

In [7]:
t=time.time();log('07','Parsing Textract JSON; reusing Drive text cache')
cache=RUN/('ocr_text_cache_layout.parquet' if USE_STRUCTURED_OCR else 'ocr_text_cache_plain.parquet')
text_map={}
if cache.exists():
 old=pd.read_parquet(cache)
 text_map=dict(zip(old.ocr_path.astype(str),old.text.astype(str)))
 log('07',f'Loaded {len(text_map)} cached OCR texts')
def textract_text(obj):
    # WikiDT compact format: [[page, [[text, [x,y,w,h]], ...]], ...]
    if isinstance(obj,list):
        collected=[]
        for page in obj:
            if isinstance(page,list) and len(page)==2 and isinstance(page[1],list):
                for element in page[1]:
                    if isinstance(element,list) and element and isinstance(element[0],str) and element[0].strip():
                        coords=element[1] if len(element)>1 else None
                        if isinstance(coords,(list,tuple)) and len(coords)>=4 and all(isinstance(v,(int,float)) for v in coords[:4]):
                            collected.append((float(coords[1]),float(coords[0]),str(element[0]),max(1,float(coords[3]))))
                        else:collected.append((None,None,str(element[0]),None))
        if collected:
            if USE_STRUCTURED_OCR and all(z[0] is not None for z in collected):
                # Approximate rows; do not infer columns or header-to-cell associations.
                median_h=float(np.median([z[3] for z in collected])); threshold=max(3,median_h*0.55)
                groups=[]
                for y,x,t,h in sorted(collected,key=lambda z:(z[0],z[1])):
                    if not groups or abs(y-groups[-1][0])>threshold:groups.append([y,[(x,t)]])
                    else:groups[-1][1].append((x,t))
                return '\n'.join(' | '.join(t for x,t in sorted(items)) for y,items in groups)
            return '\n'.join(t for y,x,t,h in collected)
        return '\n'.join(v for item in obj if (v:=textract_text(item)))
    if isinstance(obj,dict):
        blocks=obj.get('Blocks',obj.get('blocks'))
        if isinstance(blocks,list):
            lines=[b for b in blocks if isinstance(b,dict) and str(b.get('BlockType','')).upper()=='LINE' and b.get('Text')]
            if lines:return '\n'.join(str(b['Text']) for b in lines)
            words=[str(b.get('Text','')) for b in blocks if isinstance(b,dict) and str(b.get('BlockType','')).upper()=='WORD' and b.get('Text')]
            if words:return ' '.join(words)
        for k in ('textract','response','result','data'):
            if isinstance(obj.get(k),(dict,list)):
                v=textract_text(obj[k])
                if v:return v
        if isinstance(obj.get('text'),str):return obj['text']
    return ''
for p,loc in tqdm(local_paths.items(),desc='Parse OCR'):
 if p in text_map and text_map[p].strip():continue
 try:text_map[p]=textract_text(json.loads(loc.read_text(encoding='utf-8')))
 except Exception as e:log('07',f'ERROR parsing {p}: {e}');text_map[p]=''
pd.DataFrame([{'ocr_path':k,'text':v} for k,v in text_map.items()]).to_parquet(cache,index=False)
log('07',f'Nonempty={sum(bool(v.strip()) for v in text_map.values())}/{len(wanted)} requested; elapsed={time.time()-t:.1f}s')
print('[07] Example OCR TEXT (X context source):',repr(next((v[:600] for v in text_map.values() if v.strip()),'NONE')))
if not any(v.strip() for v in text_map.values()):raise RuntimeError('OCR parser produced no text; inspect JSON sample in cell 6.')

[07][03:20:45] Parsing Textract JSON; reusing Drive text cache


Parse OCR:   0%|          | 0/1709 [00:00<?, ?it/s]

[07][03:20:47] Nonempty=1694/1709 requested; elapsed=2.1s
[07] Example OCR TEXT (X context source): 'Full name | Milwaukee Rampage\nNickname(s) | Rampage\nFounded | 1993\nDissolved | 2002 (as professional franchise)\nGround | Uihlein Soccer Park Milwaukee Wisconsin\nCapacity | 7,000\nLeague | A-League\nPat Clemens | 1994\nJim Harwood | 1994-2002\nYear | Division | League | Reg. Season | Playoffs | Open Cup | Avg. Attendance\n1993 | Exhibition schedule\n1994 | 3 | USISL | 2nd, Midwest | Divisional Finals | Did not enter\n1995 | 3 | USISL Pro League | 1st, Midwest West | Sizzling Nine | Did not qualify\n1996 | 3 | USISL Select League | 1st, Central | 2nd Round | Did not qualify | 3,053\n1997 | 2 | USISLA-Lea'


## 4 — Export independent OCR passage embeddings

For each OCR JSON file, we chunk the recognized text into passages and compute an **L2-normalized semantic vector**. These vectors are reusable outside BERT QA. The artifact consists of:

- `ocr_passage_vectors.npy`: float32 matrix `[num_passages, 384]` (MiniLM output dimensionality)
- `ocr_passage_index.parquet`: mapping each vector row to `ocr_path`, passage text, and chunk number
- `ocr_embedding_manifest.json`: model, dimension, text hash, normalization and version metadata

**Contract:** A consumer must use the *same encoder* to embed new queries; arbitrary BERT inputs cannot directly use MiniLM's 384-D embeddings. For a BERT QA reader, retrieve passage text using vectors and then tokenize text with BERT. This separation permits swapping the final reader.

In [8]:
from sentence_transformers import SentenceTransformer
encoder=SentenceTransformer(EMBED_MODEL,device=DEVICE)
def chunks(txt):
    words=str(txt).split()
    return [' '.join(words[i:i+CHUNK_WORDS]) for i in range(0,len(words),CHUNK_STEP) if words[i:i+CHUNK_WORDS]]
passages=[{'ocr_path':p,'chunk_id':i,'context':c} for p in sorted(text_map) for i,c in enumerate(chunks(text_map[p]))]
passage_df=pd.DataFrame(passages)
assert len(passage_df)>0,'No OCR passages'
passage_df['row_id']=np.arange(len(passage_df))
passage_hash=hashlib.sha256(('\n'.join(passage_df.ocr_path+'\t'+passage_df.context)).encode()).hexdigest()
embed_meta={'encoder':EMBED_MODEL,'hash':passage_hash,'chunk_words':CHUNK_WORDS,'chunk_step':CHUNK_STEP,'layout':USE_STRUCTURED_OCR,'normalized':True}
embed_key=hashlib.sha256(json.dumps(embed_meta,sort_keys=True).encode()).hexdigest()[:16]
embed_dir=RUN/'embeddings'/embed_key;embed_dir.mkdir(parents=True,exist_ok=True)
vec_path=embed_dir/'ocr_passage_vectors.npy';index_path=embed_dir/'ocr_passage_index.parquet'
if vec_path.exists() and index_path.exists() and (embed_dir/'ocr_embedding_manifest.json').exists() and json.loads((embed_dir/'ocr_embedding_manifest.json').read_text())==embed_meta:
    passage_vectors=np.load(vec_path)
    passage_df=pd.read_parquet(index_path)
    log('embed','Loaded cached passage embeddings')
else:
    passage_vectors=encoder.encode(passage_df.context.tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True).astype('float32')
    np.save(vec_path,passage_vectors);passage_df.to_parquet(index_path,index=False)
    (embed_dir/'ocr_embedding_manifest.json').write_text(json.dumps(embed_meta,indent=2))
assert passage_vectors.shape[0]==len(passage_df)
assert passage_vectors.shape[1]==encoder.get_sentence_embedding_dimension()
print('Reusable embedding matrix:',passage_vectors.shape,'saved to',embed_dir)
print('Sample mapping:');display(passage_df.head(3))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (5 outputs left out)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Reusable embedding matrix: (10021, 384) saved to /content/drive/MyDrive/WikiDT_Persistent/phase7_modular_ocr_embeddings/semantic_hybrid_v1/pilot_300_60_60/embeddings/d36b4e82551979b6
Sample mapping:


/tmp/ipykernel_3553/1327219271.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert passage_vectors.shape[1]==encoder.get_sentence_embedding_dimension()


,ocr_path,chunk_id,context,row_id
0,ocr/textract/1009527_page_seg_0.json,0,Full name | Milwaukee Rampage Nickname(s) | Ra...,0
1,ocr/textract/1009527_page_seg_0.json,1,"USISL Select League | 1st, Central | 2nd Round...",1
2,ocr/textract/1009527_page_seg_0.json,2,"3rd Round | 2,682 2002 | 2 | USL A-League | 1s...",2


## 5 — Pluggable retriever: lexical + semantic fusion

Use TF-IDF question matching and MiniLM semantic similarity on the *same candidate passages*. Each score is normalized within a question's candidate set, then combined using `SEMANTIC_WEIGHT` and `LEXICAL_WEIGHT`. This is **question-only retrieval**, with no gold answer used to rank. Top-K passage texts feed the downstream QA reader. Compare answer coverage against Phase 6 to test whether semantic retrieval helps.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
path_rows=defaultdict(list)
for j,p in enumerate(passage_df.ocr_path):path_rows[p].append(j)
def answer_spans(context,answer):
    a=answer_text(answer).strip()
    return [(m.start(),m.end()) for m in re.finditer(re.escape(a),context,flags=re.I)] if a else []
def candidates_for(paths):
    return [{'path':str(passage_df.iloc[j].ocr_path),'context':str(passage_df.iloc[j].context)} for p in paths for j in path_rows.get(p,[])]
def _minmax(x):
    x=np.asarray(x,dtype=float)
    return (x-x.min())/(x.max()-x.min()) if len(x) and x.max()>x.min() else np.zeros_like(x)
def retrieve(q,paths,top_k=TOP_K,semantic_weight=SEMANTIC_WEIGHT):
    indices=[j for p in paths for j in path_rows.get(p,[])]
    if not indices:return []
    texts=passage_df.iloc[indices].context.astype(str).tolist()
    v=TfidfVectorizer(ngram_range=(1,2),stop_words='english',sublinear_tf=True)
    try:
        tf=v.fit_transform([q]+texts)
        lex=cosine_similarity(tf[:1],tf[1:]).ravel()
    except ValueError:lex=np.zeros(len(indices))
    qvec=encoder.encode([str(q)],normalize_embeddings=True,convert_to_numpy=True)[0]
    sem=passage_vectors[indices]@qvec
    score=semantic_weight*_minmax(sem)+(1-semantic_weight)*_minmax(lex)
    ranked=np.argsort(-score,kind='stable')[:top_k]
    return [{'path':str(passage_df.iloc[indices[i]].ocr_path),'context':texts[i],
             'score':float(score[i]),'semantic_score':float(sem[i]),'lexical_score':float(lex[i])} for i in ranked]
rows=[]
for r in tqdm(qa.itertuples(index=False),total=len(qa),desc='Phase7 retrieval audit'):
    hits=retrieve(r.question,r.ocr_files)
    full='\n'.join(text_map.get(p,'') for p in r.ocr_files)
    rows.append(dict(split=r.split,image=r.image,question=r.question,answer=answer_text(r.answer),ocr_files=r.ocr_files,
       hits_json=json.dumps(hits),ocr_present=bool(full.strip()),answer_in_ocr=bool(answer_spans(full,r.answer)),
       answer_in_topk=any(answer_spans(h['context'],r.answer) for h in hits)))
audit=pd.DataFrame(rows)
for split,g in audit.groupby('split'):
    log('retrieval',f'{split}: OCR={g.answer_in_ocr.mean():.2%}, top-{TOP_K}={g.answer_in_topk.mean():.2%}')
audit.assign(ocr_files=audit.ocr_files.map(json.dumps)).to_parquet(RUN/'retrieval_audit.parquet',index=False)
print('Example question:',audit.iloc[0].question)
print('Top passage:',json.loads(audit.iloc[0].hits_json)[0]['context'][:450])

Phase7 retrieval audit:   0%|          | 0/420 [00:00<?, ?it/s]

[retrieval][03:37:39] test: OCR=78.33%, top-6=71.67%
[retrieval][03:37:39] train: OCR=85.67%, top-6=79.33%
[retrieval][03:37:39] val: OCR=86.67%, top-6=83.33%
Example question: Old Mo's Second Spring was the film title used in nomination in what year?
Top passage: Year + (Ceremony) | Film title used in nomination | Original title | Language | Director | Result 1957 (30th) | Amina | Äméinà | Mandarin | Yuan Congmei | Not Nominated 1964 (37th) | Lovers' Rock | Qingrén shí | Mandarin | Not Nominated 1966 (39th) | The Silent Wife | Yanü qíngxin | Mandarin | Lee Hsing | Not Nominated 1972 (45th) | Execution Autumn | Qiûjué | Mandarin | Lee Hsing | Not Nominated 1976 (49th) | Eight Hundred Heroes | Bâbäi zhungsh


## 6 — Reader interface (swappable downstream model)

The downstream reader is isolated in separate training and prediction cells. You can swap compatible Hugging Face extractive QA checkpoints by changing `MODEL_ID`; generative readers require implementing a different prediction/training interface. A generative model requires a different reader adapter; **do not feed MiniLM embeddings into BERT's `inputs_embeds`**, because the embedding dimensions and training spaces differ.

For stronger supervision, training uses gold-positive OCR chunks (only on training split) plus retrieved hard negatives. The answer is never used to rank validation or test passages.

### Build positive and negative span training features

In [10]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
assert tokenizer.is_fast

def encode_passage(question,context,answer=None,is_positive=False):
    enc=tokenizer(str(question),context,max_length=MAX_LEN,truncation='only_second',stride=DOC_STRIDE,
                  return_overflowing_tokens=True,return_offsets_mapping=True,padding='max_length')
    golds=answer_spans(context,answer) if is_positive else []
    out=[]
    for j in range(len(enc['input_ids'])):
        offsets=enc['offset_mapping'][j]; seq=enc.sequence_ids(j)
        valid=[i for i,s in enumerate(seq) if s==1 and offsets[i][1]>offsets[i][0]]
        if not valid:continue
        rec={k:enc[k][j] for k in ('input_ids','attention_mask','token_type_ids') if k in enc}
        rec.update(context=context,offsets=offsets,sequence_ids=seq)
        cls=enc['input_ids'][j].index(tokenizer.cls_token_id)
        start=end=cls
        if is_positive:
            for lo,hi in golds:
                if offsets[valid[0]][0]<=lo and offsets[valid[-1]][1]>=hi:
                    st=next((i for i in valid if offsets[i][0]<=lo<offsets[i][1]),None)
                    en=next((i for i in reversed(valid) if offsets[i][0]<hi<=offsets[i][1]),None)
                    if st is not None and en is not None:
                        start,end=st,en;break
            if start==cls:continue # never mislabel a truncated positive window as positive
        rec['start_positions']=start;rec['end_positions']=end
        out.append(rec)
    return out

train=[];pos_questions=0;neg_windows=0
for r in tqdm(audit[audit.split=='train'].itertuples(index=False),desc='Phase 6 supervised features'):
    # Only training examples may use answer-aware oracle passages.
    all_chunks=candidates_for(r.ocr_files)
    positive=[h for h in all_chunks if answer_spans(h['context'],r.answer)]
    positive=positive[:3]  # cap duplicate passages per question
    p=[]
    for h in positive:p+=encode_passage(r.question,h['context'],r.answer,True)
    if p:pos_questions+=1;train+=p
    negatives=[h for h in json.loads(r.hits_json) if not answer_spans(h['context'],r.answer)]
    for h in negatives[:NEGATIVE_PER_QUESTION]:
        neg=encode_passage(r.question,h['context'],is_positive=False)
        train+=neg[:1];neg_windows+=min(1,len(neg))
assert train and pos_questions>0,'No positive training examples. Inspect OCR matching.'
log('09',f'Questions with positive spans={pos_questions}/{SIZES["train"]}; windows={len(train)}; hard-negative windows={neg_windows}')
example=next(x for x in train if x['start_positions']!=x['input_ids'].index(tokenizer.cls_token_id))
print('Example X context:',example['context'][:450]);print('Y token positions:',example['start_positions'],example['end_positions'])
print('Y extracted:',repr(example['context'][example['offsets'][example['start_positions']][0]:example['offsets'][example['end_positions']][1]]))


tokenizer_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Phase 6 supervised features: 0it [00:00, ?it/s]

[09][03:37:42] Questions with positive spans=257/300; windows=1104; hard-negative windows=526
Example X context: Name | Designation | Constellation | Date sent | Arrival date | Message Gliese 526 | HD 119850 | Boötes | July 10, 2013 | 2031 | Lone Signal Name | Designation HD | Constellation | Date sent | Arrival date | Message 16 Cyg A | HD 186408 | Cygnus | May 24, 1999 | November 2069 | Cosmic Call 1 15 Sge | HD 190406 | Sagitta | June 30, 1999 | February 2057 | Cosmic Call 1 HD 178428 | Sagitta | June 30, 1999 | October 2067 | Cosmic Call 1 GI 777 | HD 1
Y token positions: 186 189
Y extracted: 'July 6, 2003'


### Extractive reader prediction and metrics

In [11]:
def predict(row,max_answer_tokens=25):
    model.eval();best=(-float('inf'),'');best_null=-float('inf')
    with torch.no_grad():
        for hit in json.loads(row.hits_json):
            fs=encode_passage(row.question,hit['context'],is_positive=False)
            for f in fs:
                x={k:torch.tensor([f[k]],device=DEVICE) for k in ('input_ids','attention_mask','token_type_ids') if k in f}
                out=model(**x);a=out.start_logits[0].float().cpu().numpy();b=out.end_logits[0].float().cpu().numpy()
                cls=f['input_ids'].index(tokenizer.cls_token_id)
                best_null=max(best_null,float(a[cls]+b[cls]))
                valid=[i for i,s in enumerate(f['sequence_ids']) if s==1 and f['offsets'][i][1]>f['offsets'][i][0]]
                for ix,i in enumerate(valid):
                    for j in valid[ix:ix+max_answer_tokens]:
                        lo,hi=f['offsets'][i][0],f['offsets'][j][1]
                        cand=f['context'][lo:hi].strip()
                        score=float(a[i]+b[j])
                        if cand and score>best[0]:best=(score,cand)
    return best[1],best[0],best_null

def token_f1(p,g):
    a=norm(p).split();b=norm(g).split()
    if not a or not b:return float(a==b)
    overlap=sum((collections.Counter(a)&collections.Counter(b)).values())
    return 2*overlap/(len(a)+len(b))

def evaluate(split,save=True,verbose=True):
    d=audit[audit.split==split].copy()
    predictions=[predict(r) for r in tqdm(d.itertuples(index=False),total=len(d),desc='Evaluate '+split,leave=False)]
    d['prediction']=[p[0] for p in predictions]
    d['span_score']=[p[1] for p in predictions]
    d['null_score']=[p[2] for p in predictions]
    d['EM']=[norm(p)==norm(g) for p,g in zip(d.prediction,d.answer)]
    d['F1']=[token_f1(p,g) for p,g in zip(d.prediction,d.answer)]
    if save:d.assign(ocr_files=d.ocr_files.map(json.dumps)).to_parquet(RUN/f'{split}_predictions.parquet',index=False)
    if verbose:log('eval',f'{split}: EM={d.EM.mean():.2%}, F1={d.F1.mean():.2%}, OCR={d.answer_in_ocr.mean():.2%}, topK={d.answer_in_topk.mean():.2%}')
    return d


### Train the downstream reader with validation checkpoint selection

In [ ]:
from transformers import get_linear_schedule_with_warmup
import math
class SpanDataset(Dataset):
    def __len__(self):return len(train)
    def __getitem__(self,i):
        r=train[i]
        return {k:torch.tensor(r[k],dtype=torch.long) for k in ('input_ids','attention_mask','token_type_ids','start_positions','end_positions') if k in r}
loader=DataLoader(SpanDataset(),batch_size=BATCH_SIZE,shuffle=True)
ckpt=RUN/'best_reader_qa'
training_signature=hashlib.sha256(json.dumps({'qa':qa[['split','image','question','answer']].to_dict('records'),
 'model':MODEL_ID,'embedding_key':embed_key,'layout':USE_STRUCTURED_OCR,'semantic_weight':SEMANTIC_WEIGHT,'sizes':SIZES,'epochs':EPOCHS,'lr':LR,'top_k':TOP_K,'chunk':CHUNK_WORDS,'step':CHUNK_STEP,
 'negatives':NEGATIVE_PER_QUESTION,'grad_accum':GRAD_ACCUM,'seed':SEED},sort_keys=True,default=str).encode()).hexdigest()
meta_file=ckpt/'phase7_signature.json'
can_reuse=(ckpt/'config.json').exists() and meta_file.exists() and json.loads(meta_file.read_text()).get('sha256')==training_signature
history=[]
if can_reuse and not FORCE_RETRAIN:
    model=AutoModelForQuestionAnswering.from_pretrained(str(ckpt)).to(DEVICE)
    log('11','Loaded matching Phase 7 reader checkpoint')
else:
    model=AutoModelForQuestionAnswering.from_pretrained(MODEL_ID).to(DEVICE)
    opt=AdamW(model.parameters(),lr=LR,weight_decay=0.01)
    steps_per_epoch=math.ceil(len(loader)/GRAD_ACCUM)
    total_steps=steps_per_epoch*EPOCHS
    scheduler=get_linear_schedule_with_warmup(opt,num_warmup_steps=max(1,int(total_steps*WARMUP_RATIO)),num_training_steps=total_steps)
    best=(-1.,-1.);stale=0
    for ep in range(EPOCHS):
        model.train();opt.zero_grad(set_to_none=True);losses=[]
        for ix,batch in enumerate(tqdm(loader,desc=f'Train epoch {ep+1}/{EPOCHS}'),1):
            batch={k:v.to(DEVICE) for k,v in batch.items()}
            loss=model(**batch).loss
            (loss/GRAD_ACCUM).backward();losses.append(float(loss.detach().cpu()))
            if ix%GRAD_ACCUM==0 or ix==len(loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step();scheduler.step();opt.zero_grad(set_to_none=True)
        v=evaluate('val',save=False)
        score=(float(v.EM.mean()),float(v.F1.mean()))
        history.append({'epoch':ep+1,'train_loss':float(np.mean(losses)),'val_EM':score[0],'val_F1':score[1]})
        log('11',f'Epoch {ep+1}: loss={history[-1]["train_loss"]:.4f}, val EM={score[0]:.2%}, F1={score[1]:.2%}')
        if score>best:
            best=score;stale=0;ckpt.mkdir(parents=True,exist_ok=True)
            model.save_pretrained(str(ckpt));tokenizer.save_pretrained(str(ckpt))
            meta_file.write_text(json.dumps({'sha256':training_signature,'best_epoch':ep+1,'best_val_EM':score[0],'best_val_F1':score[1]},indent=2))
            log('11','Saved new best checkpoint')
        else:
            stale+=1
            if stale>=PATIENCE:
                log('11','Early stopping triggered');break
    pd.DataFrame(history).to_csv(RUN/'training_history.csv',index=False)
    model=AutoModelForQuestionAnswering.from_pretrained(str(ckpt)).to(DEVICE)
log('11',f'Best checkpoint ready: {ckpt}')

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at deepset/bert-base-cased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Train epoch 1/4:   0%|          | 0/276 [00:00<?, ?it/s]

Evaluate val:   0%|          | 0/60 [00:00<?, ?it/s]

[eval][04:44:35] val: EM=16.67%, F1=20.34%, OCR=86.67%, topK=83.33%
[11][04:44:35] Epoch 1: loss=2.4509, val EM=16.67%, F1=20.34%
[11][04:44:37] Saved new best checkpoint


Train epoch 2/4:   0%|          | 0/276 [00:00<?, ?it/s]

### Final held-out evaluation

In [ ]:
validation=evaluate('val')
test=evaluate('test')
summary=[]
for name,d in [('val',validation),('test',test)]:
    covered=d[d.answer_in_topk]
    summary.append({'phase':'Phase 7','split':name,'N':len(d),'EM':float(d.EM.mean()),'F1':float(d.F1.mean()),
      'OCR_coverage':float(d.answer_in_ocr.mean()),'retrieval_coverage':float(d.answer_in_topk.mean()),
      'conditional_EM':float(covered.EM.mean()) if len(covered) else float('nan')})
summary_df=pd.DataFrame(summary)
summary_df.to_csv(RUN/'phase7_metrics.csv',index=False)
display(summary_df)
print('Historical Phase 6: val EM 13.33%, F1 17.22%; test EM 6.67%, F1 11.06% (compare only if manifests match).')
print('Phase 7 model output:',RUN)
print('Validation reader errors despite answer in retrieved text:',int(((~validation.EM)&validation.answer_in_topk).sum()))
display(validation[['question','answer','prediction','EM','F1','answer_in_ocr','answer_in_topk']].head(20))

## 7 — Export contract and examples for other downstream models

This block demonstrates a **standalone retrieval interface** returning both vectors and text. A classifier can consume `passage_vectors` directly. A QA model like BERT consumes `passage_text` and the question. The metadata identifies the embedding model and source passage.

**Optional experiment:** train a downstream answer-type classifier on the vectors or replace the reader with a different pretrained QA model. Keep the same test set and never select model parameters based on test results.

In [ ]:
def get_ocr_embeddings(ocr_paths):
    """Return normalized MiniLM passage vectors and their text/metadata; no QA model needed."""
    indices=[j for p in ocr_paths for j in path_rows.get(p,[])]
    return passage_vectors[indices],passage_df.iloc[indices].reset_index(drop=True)
def get_retrieved_evidence(question,ocr_paths,k=TOP_K):
    """Portable evidence interface for any downstream text-based QA model."""
    return retrieve(question,ocr_paths,top_k=k)
example_row=qa.iloc[0]
vectors,metadata=get_ocr_embeddings(example_row.ocr_files)
print('Independent OCR embedding tensor:',vectors.shape)
print('Top evidence for downstream QA:')
for hit in get_retrieved_evidence(example_row.question,example_row.ocr_files,k=2):
    print('score=',round(hit['score'],3),'text=',hit['context'][:240])
print('Embedding metadata:',embed_dir/'ocr_embedding_manifest.json')
print('QA reader checkpoint:',ckpt)
print('Evaluation:',RUN/'phase7_metrics.csv')

## 8 — Interpreting Phase 7 results

1. **OCR coverage** measures whether the answer text exists in the recognized OCR, independent of retrieval.
2. **Top-K coverage** measures whether question-only retrieval found an answer-bearing passage.
3. **Conditional EM** measures answer extraction accuracy when the answer was available to the reader.
4. **Validation/test EM and token F1** measure end-to-end performance.
5. **Same-cohort comparison:** inspect the Phase 6 and Phase 7 manifests; do not attribute differences to model changes if sampled questions differ.

### Limitations
- MiniLM is a semantic *text encoder*, not a trainable image OCR model. The OCR stage remains precomputed Textract.
- Bounding-box row grouping is approximate and can misorder multi-column documents; compare `USE_STRUCTURED_OCR=False` as an ablation.
- Extractive QA cannot produce answers absent from retrieved OCR text.
- Training on oracle-positive passages is a training technique, **not** a valid way to select validation/test evidence.
- Hard-negative passages and small datasets can still cause overfitting; no performance gain is guaranteed.

**Suggested ablations:** TF-IDF-only vs semantic-only vs hybrid; flattened vs row-grouped OCR; BERT vs another QA checkpoint. Run each with fixed manifest and report all results.